In [ ]:
# Cell 1：最初はこのセルだけ実行し、入力欄を自分の値に変更してください。
# SET_ / REPLACE_WITH_ は未設定です。既存Widgetは再作成しません。
defaults = {
    "work_root": "/Volumes/REPLACE_WITH_CATALOG/REPLACE_WITH_SCHEMA/REPLACE_WITH_WORK_VOLUME",
    "source_run_id_2020": "SET_SOURCE_RUN_ID_2020",
    "source_run_id_2025": "SET_SOURCE_RUN_ID_2025",
    "comparison_run_id": "SET_NEW_COMPARISON_RUN_ID",
    "boundary_policy": "strict"
}
labels = {
    "work_root": "成果物Volumeの絶対パス（4本で共通）",
    "source_run_id_2020": "nb01の2020年PASS済みID",
    "source_run_id_2025": "nb01の2025年PASS済みID",
    "comparison_run_id": "比較run ID：nb02は未使用、nb03は61件PASS済みID",
    "boundary_policy": "初回strict／根拠確認後exclude_documented_machida"
}
for name, value in defaults.items():
    try:
        aidputils.widgets.get(name)
    except Exception:
        aidputils.widgets.text(name=name, defaultValue=value, label=labels[name])
for name in defaults:
    print(name, '=', aidputils.widgets.get(name))


# nb02_compare_years — 2020→2025 の比較と境域検査
両年の原数値を検査し、Sparkで増減を計算する。strictで差分を確認した後、根拠を確認して
boundary_policy=exclude_documented_machida と新しいcomparison_run_idを設定すると、町田市を除く61市区町村を分析対象にする。
人口だけの組替では世帯・年齢の境域を揃えられない。原本・失敗した実行は上書きしない。


In [ ]:
import json, math, re
from pathlib import Path
from datetime import datetime, timezone
from pyspark.sql import functions as F

PARAMETERS = {
    'work_root': '/Volumes/REPLACE_WITH_CATALOG/REPLACE_WITH_SCHEMA/REPLACE_WITH_WORK_VOLUME',
    'source_run_id_2020': 'SET_SOURCE_RUN_ID_2020',
    'source_run_id_2025': 'SET_SOURCE_RUN_ID_2025',
    'comparison_run_id': 'SET_NEW_COMPARISON_RUN_ID',
    'boundary_policy': 'strict',
}
for key, value in PARAMETERS.items():
    try:
        aidputils.widgets.get(key)
    except Exception:
        aidputils.widgets.text(name=key, defaultValue=value, label=key)
P = {key: aidputils.widgets.get(name=key).strip() for key in PARAMETERS}
for key in ['source_run_id_2020','source_run_id_2025','comparison_run_id']:
    if P[key].startswith('SET_') or not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,99}', P[key]):
        raise ValueError('Invalid run identifier: ' + key)
WORK = Path(P['work_root'])
if ('REPLACE_WITH_' in str(WORK) or not str(WORK).startswith('/Volumes/') or '..' in WORK.parts):
    raise ValueError('work_root を実際の /Volumes/... 絶対パスに設定してください')
ROOT = WORK / 'comparisons' / P['comparison_run_id']
ROOT.mkdir(parents=True, exist_ok=False)
print({k: v for k, v in P.items() if k != 'work_root'})


## 検査関数
このセルは同梱の comparison_core.py と同じ内容。Notebook単体で実行できる。

In [ ]:
"""Small-data validation and a local reference calculation; Spark computes production metrics."""
import hashlib
import json
import math

TOKYO_CODES = {str(x) for x in range(13101, 13124)} | {
    str(x) for x in [13201,13202,13203,13204,13205,13206,13207,13208,13209,
    13210,13211,13212,13213,13214,13215,13218,13219,13220,13221,13222,
    13223,13224,13225,13227,13228,13229,13303,13305,13307,13308,13361,
    13362,13363,13364,13381,13382,13401,13402,13421]}
BOUNDARY_REASON = {
    'area_code': '13209', 'area_name': '町田市',
    'reason': '2020-12-01境界変更。人口だけを組み替えても世帯・年齢の比較条件は揃わないため除外。',
    'effective_date': '2020-12-01',
    'source_url': 'https://www.city.sagamihara.kanagawa.jp/shisei/1026766/seido/1026777/1004437.html',
    'supporting_url': 'https://www.gikai-machida.jp/voices/GikaiDoc/attach/Gk/Gk5001_64_somu_gyosei_kyokaihenko.pdf'
}
COUNT_FIELDS = ['population','general_households','single_households',
                'age_under15','age_15_64','age_65plus']

def validate_year_records(records, year):
    if len(records) != 62 or {r['area_code'] for r in records} != TOKYO_CODES:
        raise ValueError(f'{year}: current Tokyo municipality coverage must be exactly 62 unique codes')
    for r in records:
        if r['census_year'] != year or r['series'] != 'original' or r['reference_date'] != f'{year}-10-01':
            raise ValueError(f'{year}: wrong year, reference date or series')
        for key in COUNT_FIELDS:
            if type(r[key]) is not int or r[key] < 0:
                raise ValueError(f"{r['area_code']}: invalid integer count {key}")
        known = sum(r[k] for k in ['age_under15','age_15_64','age_65plus'])
        if r['population'] <= 0 or r['general_households'] <= 0 or known <= 0:
            raise ValueError('Zero denominator')
        if r['single_households'] > r['general_households'] or known > r['population']:
            raise ValueError('Inconsistent population/household counts')
        unknown = r['age_unknown']
        if unknown is not None and (type(unknown) is not int or unknown < 0 or known + unknown != r['population']):
            raise ValueError('Age totals disagree with population')
        if year == 2025 and (type(r['population_previous_rearranged']) is not int or r.get('population_previous_year') != 2020):
            raise ValueError('2025 must contain a complete rearranged 2020 population')
    return {r['area_code']: r for r in records}

def compare_scope(old_records, new_records, boundary_policy):
    old = validate_year_records(old_records, 2020)
    new = validate_year_records(new_records, 2025)
    diffs = [{'area_code': c, 'area_name': new[c]['area_name'],
              'population_2020_original': old[c]['population'],
              'population_2020_rearranged': new[c]['population_previous_rearranged'],
              'difference': new[c]['population_previous_rearranged'] - old[c]['population']}
             for c in sorted(TOKYO_CODES)
             if new[c]['population_previous_rearranged'] != old[c]['population']]
    if boundary_policy == 'strict':
        excluded = []
        accepted = not diffs
    elif boundary_policy == 'exclude_documented_machida':
        # Deliberate scope choice backed by the cited boundary record, not a tolerance.
        excluded = ['13209']
        accepted = all(d['area_code'] == '13209' and d['difference'] == 4 for d in diffs)
    else:
        raise ValueError('Unsupported boundary_policy')
    return {'accepted': accepted, 'differences': diffs,
            'input_municipality_count': 62, 'rows': 62 - len(excluded),
            'excluded_area_codes': excluded,
            'excluded_reasons': [BOUNDARY_REASON] if excluded else [],
            'boundary_policy': boundary_policy,
            'caveat': 'Population agreement is a check, not proof that every demographic definition is unchanged.'}

def reference_changes(old_records, new_records, scope):
    if not scope['accepted']:
        raise ValueError('Unresolved boundary mismatch; no ML input may be generated')
    old = {r['area_code']: r for r in old_records}
    new = {r['area_code']: r for r in new_records}
    output = []
    for c in sorted(TOKYO_CODES - set(scope['excluded_area_codes'])):
        row = {'area_code': c, 'area_name': new[c]['area_name']}
        for year, source in [(2020, old[c]), (2025, new[c])]:
            known = source['age_under15'] + source['age_15_64'] + source['age_65plus']
            row[f'population_{year}'] = source['population']
            row[f'single_households_pct_{year}'] = source['single_households'] / source['general_households'] * 100.0
            row[f'age_65plus_share_known_pct_{year}'] = source['age_65plus'] / known * 100.0
            row[f'age_unknown_pct_{year}'] = None if source['age_unknown'] is None else source['age_unknown'] / source['population'] * 100.0
        row['population_growth_pct'] = (row['population_2025'] / row['population_2020'] - 1.0) * 100.0
        row['single_share_change_pp'] = row['single_households_pct_2025'] - row['single_households_pct_2020']
        row['elderly_share_change_pp'] = row['age_65plus_share_known_pct_2025'] - row['age_65plus_share_known_pct_2020']
        row['age_unknown_change_pp'] = (None if row['age_unknown_pct_2020'] is None or row['age_unknown_pct_2025'] is None
                                       else row['age_unknown_pct_2025'] - row['age_unknown_pct_2020'])
        output.append(row)
    return output

def canonical_hash(records):
    return hashlib.sha256(json.dumps(sorted(records, key=lambda r: r['area_code']),
        ensure_ascii=False, sort_keys=True, allow_nan=False, separators=(',', ':')).encode()).hexdigest()


In [ ]:
sources = {}
source_meta = {}
source_records = {}
for year in [2020, 2025]:
    source_root = WORK / 'runs' / P[f'source_run_id_{year}']
    target = str(source_root / 'curated' / 'tokyo_year')
    meta = json.loads((source_root / 'validation.json').read_text())
    if (meta.get('status') != 'PASS' or meta.get('census_year') != year
            or meta.get('rows') != 62 or meta.get('target') != target
            or meta.get('series') != 'original'
            or meta.get('unknown_policy') != 'preserve_dash_as_null'):
        raise ValueError(f'{year}: upstream validation is absent or incompatible')
    df = spark.read.parquet(target)
    records = [r.asDict(recursive=True) for r in df.orderBy('area_code').limit(63).collect()]
    validate_year_records(records, year)
    if meta.get('curated_sha256') != canonical_hash(records):
        raise ValueError(f'{year}: curated rows no longer match the upstream validation hash')
    sources[year], source_meta[year], source_records[year] = df, meta, records

scope = compare_scope(source_records[2020], source_records[2025], P['boundary_policy'])
with (ROOT/'boundary_comparison.json').open('x') as f:
    json.dump(scope, f, ensure_ascii=False, indent=2, allow_nan=False)
print(json.dumps(scope, ensure_ascii=False, indent=2))
if not scope['accepted']:
    with (ROOT/'validation.json').open('x') as f:
        json.dump({'status':'FAIL','reason':'BOUNDARY_MISMATCH',**scope},f,ensure_ascii=False,indent=2)
    raise ValueError('Boundary mismatch. Inspect boundary_comparison.json; do not change the tolerance. Use a new run ID for a documented scope change.')


## Sparkで割合と変化を計算する
高齢者割合の分母は年齢既知人口。割合の差の単位はpercentage points。年齢不詳がNULLなら、不詳率とその差もNULLのまま残す。

In [ ]:
by_year = {}
for year, df in sources.items():
    by_year[year] = df.select(
        'area_code', F.col('area_name').alias(f'area_name_{year}'),
        F.col('population').alias(f'population_{year}'),
        (F.col('single_households') / F.col('general_households') * 100.0).alias(f'single_households_pct_{year}'),
        (F.col('age_65plus') / (F.col('age_under15') + F.col('age_15_64') + F.col('age_65plus')) * 100.0).alias(f'age_65plus_share_known_pct_{year}'),
        (F.col('age_unknown') / F.col('population') * 100.0).alias(f'age_unknown_pct_{year}')
    )
changes = by_year[2020].join(by_year[2025], on='area_code', how='inner')
if scope['excluded_area_codes']:
    changes = changes.where(~F.col('area_code').isin(scope['excluded_area_codes']))
changes = (changes.withColumn('area_name', F.col('area_name_2025'))
    .withColumn('population_growth_pct', (F.col('population_2025') / F.col('population_2020') - 1.0) * 100.0)
    .withColumn('single_share_change_pp', F.col('single_households_pct_2025') - F.col('single_households_pct_2020'))
    .withColumn('elderly_share_change_pp', F.col('age_65plus_share_known_pct_2025') - F.col('age_65plus_share_known_pct_2020'))
    .withColumn('age_unknown_change_pp', F.col('age_unknown_pct_2025') - F.col('age_unknown_pct_2020'))
    .drop('area_name_2020','area_name_2025'))
reference = reference_changes(source_records[2020], source_records[2025], scope)
actual = [r.asDict() for r in changes.orderBy('area_code').limit(63).collect()]
if len(actual) != scope['rows']:
    raise ValueError('Unexpected comparison count')
for observed, expected in zip(actual, reference):
    for key, value in expected.items():
        observed_value = observed[key]
        if isinstance(value, float):
            if observed_value is None or not math.isfinite(observed_value) or not math.isclose(value, observed_value, abs_tol=1e-10, rel_tol=1e-12):
                raise ValueError(f'Calculated value mismatch: {expected["area_code"]} {key}')
        elif observed_value != value:
            raise ValueError(f'Comparison mismatch: {expected["area_code"]} {key}')
changes.orderBy('population_growth_pct',ascending=False).show(15,truncate=False)


In [ ]:
target = str(ROOT / 'curated' / 'tokyo_changes')
changes.write.mode('errorifexists').parquet(target)
changes.orderBy('area_code').coalesce(1).write.mode('errorifexists').option('header',True).csv(str(ROOT/'csv'/'tokyo_changes'))
roundtrip = [r.asDict() for r in spark.read.parquet(target).orderBy('area_code').limit(63).collect()]
if canonical_hash(roundtrip) != canonical_hash(actual):
    raise ValueError('Parquet readback differs')
with (ROOT/'comparison_snapshot.json').open('x') as f:
    json.dump(actual,f,ensure_ascii=False,indent=2,allow_nan=False)
metadata = {
    'status':'PASS', 'comparison_run_id':P['comparison_run_id'],
    'created_at':datetime.now(timezone.utc).isoformat(),
    'target':target,'series':'original','period':'2020-2025',
    'unknown_policy':'preserve_dash_as_null',
    'source_run_ids':{str(y):P[f'source_run_id_{y}'] for y in [2020,2025]},
    'source_sha256':{str(y):canonical_hash(source_records[y]) for y in [2020,2025]},
    'comparison_sha256':canonical_hash(actual),'spark_version':spark.version,
    **scope,
    'definitions':{
        'population_growth_pct':'(population_2025/population_2020-1)*100; percent over five years',
        'single_households_pct':'single/general_households*100',
        'age_65plus_share_known_pct':'age65plus/(under15+age15to64+age65plus)*100',
        'share_changes':'2025 share minus 2020 share; percentage points',
        'age_unknown_pct':'unknown/population*100; NULL preserved',
    },
}
with (ROOT/'validation.json').open('x') as f:
    json.dump(metadata,f,ensure_ascii=False,indent=2,allow_nan=False)
print(json.dumps({'status':'PASS','rows':scope['rows'],'target':target},ensure_ascii=False))
